# FETA-Transformer + PREG-Net Pregnancy Loss Project

This is a **standalone Colab-ready notebook**. You can upload only this `.ipynb` file to Google Colab and click **Runtime > Run all**. The notebook contains an embedded copy of the project source code, writes the required Python files into the Colab runtime, installs dependencies, generates the dataset, trains/evaluates the models, exports statistics, and builds figures/tables.

Default `RUN_MODE` is `quick` so the whole structure runs without a very long training job. Change `RUN_MODE` to `full` in Stage 0 for the larger 800-patient / 80-epoch / 5-fold research run.


## Stage 0 - Bootstrap The Standalone Project

This cell creates a fresh project folder inside Colab (`/content/feta_preg_project`) and extracts the embedded source files. No GitHub clone or sidecar files are required.


In [ ]:
from __future__ import annotations

import base64
import io
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def env_flag(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_MODE = os.environ.get("FETA_PREG_RUN_MODE", "quick").strip().lower()
if RUN_MODE not in {"quick", "full"}:
    raise ValueError("RUN_MODE must be 'quick' or 'full'.")

ROOT = Path("/content/feta_preg_project") if IN_COLAB else Path.cwd().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

SOURCE_ZIP_B64 = """
...
SwUGAAAAABcAFwAQBgAAWyABAAAA
"""

source_bytes = base64.b64decode("".join(SOURCE_ZIP_B64.split()))
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    root_resolved = ROOT.resolve()
    for member in archive.namelist():
        target = (ROOT / member).resolve()
        if not str(target).startswith(str(root_resolved)):
            raise RuntimeError(f"Unsafe embedded path: {member}")
    archive.extractall(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42
DEVICE = os.environ.get("FETA_PREG_DEVICE", "auto")
INSTALL_MISSING_DEPS = env_flag("FETA_PREG_INSTALL_DEPS", True)
INSTALL_OPTIONAL_DEPS = env_flag("FETA_PREG_INSTALL_OPTIONAL_DEPS", RUN_MODE == "full")
RUN_PUBLIC_DOWNLOADS = env_flag("FETA_PREG_PUBLIC_DOWNLOADS", True)
RUN_LARGE_EXTERNAL_DOWNLOADS = env_flag("FETA_PREG_LARGE_DOWNLOADS", False)
CLEAN_OUTPUTS = env_flag("FETA_PREG_CLEAN_OUTPUTS", True)

CONFIG = {
    "quick": {
        "n_patients": 300,
        "train_epochs": 3,
        "train_patience": 3,
        "baseline_epochs": 120,
        "baseline_patience": 20,
        "batch_size": 64,
        "cv_models": "preg,feta,ensemble,torch_mlp,sklearn_logistic_regression",
        "cv_folds": 2,
        "cv_max_folds": 1,
        "cv_epochs": 2,
        "cv_patience": 2,
        "bootstrap_iterations": 200,
    },
    "full": {
        "n_patients": 800,
        "train_epochs": 80,
        "train_patience": 15,
        "baseline_epochs": 300,
        "baseline_patience": 50,
        "batch_size": 32,
        "cv_models": "preg,feta,ensemble,learned_ensemble,torch_mlp,sklearn_logistic_regression,random_forest,hist_gradient_boosting,xgboost",
        "cv_folds": 5,
        "cv_max_folds": None,
        "cv_epochs": 80,
        "cv_patience": 15,
        "bootstrap_iterations": 2000,
    },
}[RUN_MODE]

DATA_DIR = ROOT / "data" / "generated"
DATASETS_DIR = ROOT / "datasets"
RESULTS_DIR = ROOT / "results"
NOTEBOOK_FIGURES_DIR = RESULTS_DIR / "notebook_figures"

print(f"Colab runtime : {IN_COLAB}")
print(f"Project root  : {ROOT}")
print(f"Run mode      : {RUN_MODE}")
print(f"Python        : {sys.version.split()[0]} on {platform.platform()}")
print(f"Initial device: {DEVICE}")
print("Embedded files written:")
for path in sorted([p for p in ROOT.rglob("*.py") if ".ipynb_checkpoints" not in str(p)]):
    print(" -", path.relative_to(ROOT))
print(json.dumps(CONFIG, indent=2))


## Stage 1 - Dependencies

Colab usually already includes PyTorch, NumPy, pandas, matplotlib, seaborn, and scikit-learn. This cell checks imports and installs anything missing. Optional packages are installed only in `full` mode by default.


In [ ]:
REQUIRED_IMPORTS = {
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}
OPTIONAL_IMPORTS = {
    "scipy": "scipy",
    "xgboost": "xgboost",
    "openpyxl": "openpyxl",
    "xlrd": "xlrd",
}


def pip_install(args: list[str]) -> None:
    quiet_prefix = ["-q"] if IN_COLAB else []
    command = [sys.executable, "-m", "pip", "install", *quiet_prefix, *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    proc = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode == 0:
        return

    if "externally-managed-environment" in proc.stdout:
        fallback = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--user",
            "--break-system-packages",
            *args,
        ]
        print("Base Python is externally managed; retrying with user-site install.")
        print("$", " ".join(shlex.quote(part) for part in fallback))
        fallback_proc = subprocess.run(fallback, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if fallback_proc.stdout:
            print(fallback_proc.stdout)
        if fallback_proc.returncode == 0:
            return
        raise RuntimeError(f"pip install failed with exit {fallback_proc.returncode}")

    raise RuntimeError(f"pip install failed with exit {proc.returncode}")


def missing_packages(import_to_package: dict[str, str]) -> list[str]:
    missing = []
    for import_name, package_name in import_to_package.items():
        try:
            __import__(import_name)
        except Exception:
            missing.append(package_name)
    return missing

missing_required = missing_packages(REQUIRED_IMPORTS)
if missing_required and INSTALL_MISSING_DEPS:
    print("Installing missing required packages:", missing_required)
    pip_install(missing_required)
elif missing_required:
    raise RuntimeError(f"Missing required packages: {missing_required}")
else:
    print("Required imports are available.")

missing_optional = missing_packages(OPTIONAL_IMPORTS)
if missing_optional and INSTALL_OPTIONAL_DEPS:
    print("Installing missing optional packages:", missing_optional)
    pip_install(missing_optional)
elif missing_optional:
    print("Optional packages not installed in this mode; affected steps will skip gracefully if needed:", missing_optional)
else:
    print("Optional imports are available.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import HTML, Image, Markdown, SVG, display

if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Resolved device: {DEVICE}")
if IN_COLAB and DEVICE == "cpu":
    print("Tip: Runtime > Change runtime type > GPU will speed up the deep-model cells.")

sns.set_theme(style="whitegrid", context="notebook")


## Stage 2 - Utility Functions And Output Reset

These helpers run project scripts, download public files, safely extract archives, and display generated artifacts. Cleaning is enabled by default so a Run All starts from a consistent state.


In [ ]:
def run_command(args: list[str | Path], check: bool = True) -> subprocess.CompletedProcess:
    cmd = [str(arg) for arg in args]
    print("\n$", " ".join(shlex.quote(part) for part in cmd))
    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    collected = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        collected.append(line)
    return_code = proc.wait()
    elapsed = time.time() - start
    print(f"[exit={return_code}, elapsed={elapsed:.1f}s]")
    completed = subprocess.CompletedProcess(cmd, return_code, "".join(collected), None)
    if check and return_code != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return completed


def reset_outputs() -> None:
    targets = [
        DATA_DIR,
        RESULTS_DIR / "preg_net",
        RESULTS_DIR / "feta_transformer",
        RESULTS_DIR / "ensemble",
        RESULTS_DIR / "ensemble_learned",
        RESULTS_DIR / "baselines",
        RESULTS_DIR / "cross_validation",
        RESULTS_DIR / "statistical_analysis",
        RESULTS_DIR / "explainability",
        RESULTS_DIR / "paper_artifacts",
        NOTEBOOK_FIGURES_DIR,
        RESULTS_DIR / "model_comparison.csv",
        RESULTS_DIR / "model_comparison.json",
        RESULTS_DIR / "model_comparison.md",
    ]
    for path in targets:
        if path.is_dir():
            shutil.rmtree(path)
            print(f"Removed directory: {path.relative_to(ROOT)}")
        elif path.exists():
            path.unlink()
            print(f"Removed file: {path.relative_to(ROOT)}")


def download_file(url: str, destination: Path, required: bool = False, timeout: int = 60) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already downloaded: {destination.relative_to(ROOT)}")
        return True
    print(f"Downloading: {url}")
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=timeout) as response, destination.open("wb") as handle:
            shutil.copyfileobj(response, handle)
        print(f"Saved: {destination.relative_to(ROOT)} ({destination.stat().st_size:,} bytes)")
        return True
    except Exception as exc:
        message = f"Download failed for {url}: {type(exc).__name__}: {exc}"
        if required:
            raise RuntimeError(message) from exc
        print(message)
        return False


def safe_extract_zip(path: Path, destination: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(path) as archive:
        for member in archive.namelist():
            target = (destination / member).resolve()
            if not str(target).startswith(str(base)):
                raise RuntimeError(f"Unsafe zip member path: {member}")
        archive.extractall(destination)
    print(f"Extracted zip: {path.relative_to(ROOT)} -> {destination.relative_to(ROOT)}")
    return True


def safe_extract_tar(path: Path, destination: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with tarfile.open(path) as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if not str(target).startswith(str(base)):
                raise RuntimeError(f"Unsafe tar member path: {member.name}")
        archive.extractall(destination)
    print(f"Extracted tar: {path.relative_to(ROOT)} -> {destination.relative_to(ROOT)}")
    return True


def display_if_exists(path: Path) -> None:
    if not path.exists():
        print(f"Missing: {path.relative_to(ROOT)}")
        return
    if path.suffix.lower() == ".svg":
        display(SVG(filename=str(path)))
    elif path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        display(Image(filename=str(path)))
    elif path.suffix.lower() in {".md", ".txt"}:
        display(Markdown(path.read_text(encoding="utf-8")))
    elif path.suffix.lower() == ".csv":
        display(pd.read_csv(path).head(20))
    elif path.suffix.lower() == ".json":
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
    else:
        print(path)

if CLEAN_OUTPUTS:
    reset_outputs()
else:
    print("CLEAN_OUTPUTS is False; existing results will be reused/overwritten as needed.")

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Stage 3 - Public Data Download And Archive Extraction

The available public files are useful as research context, but they are not all first-trimester longitudinal miscarriage cohorts with the same schema. The actual model training cohort is generated in Stage 5. Downloads are optional and failures do not stop the project pipeline.


In [ ]:
raw_dir = DATASETS_DIR / "raw"
extract_dir = DATASETS_DIR / "extracted"
raw_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)

PUBLIC_DOWNLOADS = [
    {
        "name": "Detti2020.pdf",
        "url": "https://www.nature.com/articles/s41598-020-58114-3.pdf",
        "kind": "pdf",
        "required": False,
    },
    {
        "name": "cardiotocography.zip",
        "url": "https://archive.ics.uci.edu/static/public/193/cardiotocography.zip",
        "kind": "zip",
        "required": False,
    },
]

LARGE_EXTERNAL_DOWNLOADS = [
    {
        "name": "CRL_images.zip",
        "url": "https://figshare.com/ndownloader/files/16570518",
        "kind": "zip",
        "required": False,
    },
]

download_manifest = []
if RUN_PUBLIC_DOWNLOADS:
    for item in PUBLIC_DOWNLOADS:
        dest = raw_dir / item["name"]
        ok = download_file(item["url"], dest, required=item.get("required", False))
        download_manifest.append({**item, "path": str(dest), "downloaded": ok})
else:
    print("RUN_PUBLIC_DOWNLOADS is False; skipping lightweight public downloads.")

if RUN_LARGE_EXTERNAL_DOWNLOADS:
    for item in LARGE_EXTERNAL_DOWNLOADS:
        dest = raw_dir / item["name"]
        ok = download_file(item["url"], dest, required=item.get("required", False), timeout=120)
        download_manifest.append({**item, "path": str(dest), "downloaded": ok})
else:
    print("RUN_LARGE_EXTERNAL_DOWNLOADS is False; skipping large optional archives.")

for item in download_manifest:
    path = Path(item["path"])
    if not item["downloaded"]:
        continue
    if item["kind"] == "zip":
        safe_extract_zip(path, extract_dir / path.stem)
    elif item["kind"] in {"tar", "tgz", "tar.gz"}:
        safe_extract_tar(path, extract_dir / path.stem)

manifest_path = DATASETS_DIR / "download_manifest.json"
manifest_path.write_text(json.dumps(download_manifest, indent=2), encoding="utf-8")
display(pd.DataFrame(download_manifest) if download_manifest else pd.DataFrame())


## Stage 4 - External Data Inspection And Schema Notes

This cell inspects any downloaded public datasets that can be parsed locally. The downstream models require patient-level first-trimester longitudinal ultrasound rows with `FHR`, `CRL`, `GS`, `YSD`, gestational age, maternal features, and pregnancy-loss labels. Public CTG datasets are later-pregnancy fetal-monitoring data, so they are not merged into the final AVM modeling cohort.


In [ ]:
external_summaries = []
ctg_candidates = list(extract_dir.rglob("CTG.xls")) + list(extract_dir.rglob("*.xls"))
ctg_df = pd.DataFrame()

for candidate in ctg_candidates[:3]:
    try:
        parsed = pd.read_excel(candidate)
        if len(parsed) > 0:
            ctg_df = parsed.copy()
            external_summaries.append({
                "source": "UCI Cardiotocography",
                "path": str(candidate.relative_to(ROOT)),
                "rows": int(parsed.shape[0]),
                "columns": int(parsed.shape[1]),
                "note": "Parsed successfully; not schema-compatible with first-trimester longitudinal AVM cohort.",
            })
            break
    except Exception as exc:
        external_summaries.append({
            "source": "UCI Cardiotocography",
            "path": str(candidate.relative_to(ROOT)),
            "rows": None,
            "columns": None,
            "note": f"Could not parse locally: {type(exc).__name__}: {exc}",
        })

local_pdfs = [path for path in [ROOT / "Research.pdf", ROOT / "Base-Paper.pdf", raw_dir / "Detti2020.pdf"] if path.exists()]
for path in local_pdfs:
    external_summaries.append({
        "source": path.name,
        "path": str(path.relative_to(ROOT)),
        "rows": None,
        "columns": None,
        "note": "Reference document available for project rationale.",
    })

external_summary_df = pd.DataFrame(external_summaries)
external_summary_df.to_csv(DATASETS_DIR / "external_data_summary.csv", index=False)
display(external_summary_df)
if not ctg_df.empty:
    display(ctg_df.head())


## Stage 5 - Synthetic Cohort Generation, Cleaning, And Validation

The synthetic generator creates a first-trimester longitudinal cohort with biologically motivated trajectories, patient-level maternal risk features, clinically plausible missingness, and post-generation imputation. It writes the canonical `patients.csv`, `scans.csv`, and `dataset_summary.json` used by every downstream script.


In [ ]:
run_command([
    sys.executable,
    "synthetic_generator_v2.py",
    "--out",
    DATA_DIR,
    "--seed",
    SEED,
    "--n-patients",
    CONFIG["n_patients"],
])

patients_df = pd.read_csv(DATA_DIR / "patients.csv")
scans_df = pd.read_csv(DATA_DIR / "scans.csv")

required_patient_cols = {"patient_id", "age", "bmi", "parity", "gravidity", "previous_loss", "conception", "singleton", "label"}
required_scan_cols = {"patient_id", "gestational_age_weeks", "FHR", "CRL", "GS", "YSD"}
missing_patient = required_patient_cols - set(patients_df.columns)
missing_scan = required_scan_cols - set(scans_df.columns)
if missing_patient or missing_scan:
    raise ValueError({"missing_patient_columns": sorted(missing_patient), "missing_scan_columns": sorted(missing_scan)})

patients_df = patients_df.drop_duplicates("patient_id").copy()
patients_df["label"] = patients_df["label"].astype(int)
patients_df["singleton"] = patients_df["singleton"].astype(bool)
for col in ["age", "bmi", "parity", "gravidity", "previous_loss"]:
    patients_df[col] = pd.to_numeric(patients_df[col], errors="coerce")

scans_df = scans_df.drop_duplicates().sort_values(["patient_id", "gestational_age_weeks"]).copy()
for col in ["gestational_age_weeks", "FHR", "CRL", "GS", "YSD"]:
    scans_df[col] = pd.to_numeric(scans_df[col], errors="coerce")

# Defensive cleaning in case a future generator or imported cohort leaves residual missing values.
ultrasound_cols = ["FHR", "CRL", "GS", "YSD"]
missing_before = scans_df[ultrasound_cols].isna().sum().to_dict()
for col in ultrasound_cols:
    scans_df[col] = scans_df.groupby("patient_id")[col].transform(lambda values: values.interpolate().bfill().ffill())
    scans_df[col] = scans_df[col].fillna(scans_df[col].median())
missing_after = scans_df[ultrasound_cols].isna().sum().to_dict()

patients_df.to_csv(DATA_DIR / "patients.csv", index=False)
scans_df.to_csv(DATA_DIR / "scans.csv", index=False)

scans_per_patient = scans_df.groupby("patient_id").size()
validation_summary = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "n_patients": int(patients_df["patient_id"].nunique()),
    "n_scans": int(len(scans_df)),
    "loss_rate": float(patients_df["label"].mean()),
    "scans_per_patient_min": int(scans_per_patient.min()),
    "scans_per_patient_max": int(scans_per_patient.max()),
    "scans_per_patient_mean": float(scans_per_patient.mean()),
    "gestational_age_min": float(scans_df["gestational_age_weeks"].min()),
    "gestational_age_max": float(scans_df["gestational_age_weeks"].max()),
    "missing_before_defensive_cleaning": missing_before,
    "missing_after_defensive_cleaning": missing_after,
    "all_patients_have_scans": bool(scans_df["patient_id"].nunique() == patients_df["patient_id"].nunique()),
}
(DATA_DIR / "notebook_validation_summary.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")

patient_json_dir = DATA_DIR / "patient_json"
patient_json_dir.mkdir(parents=True, exist_ok=True)
for _, row in patients_df.iterrows():
    pid = row["patient_id"]
    patient_scans = scans_df[scans_df["patient_id"] == pid].sort_values("gestational_age_weeks")
    record = {
        "patient_id": pid,
        "label": int(row["label"]),
        "maternal": {
            "age": float(row["age"]),
            "bmi": float(row["bmi"]),
            "gravidity": int(row["gravidity"]),
            "parity": int(row["parity"]),
            "previous_loss": int(row["previous_loss"]),
        },
        "pregnancy": {
            "conception": str(row["conception"]),
            "singleton": bool(row["singleton"]),
        },
        "ultrasounds": patient_scans[["gestational_age_weeks", "FHR", "CRL", "GS", "YSD"]].to_dict(orient="records"),
        "outcome": {
            "miscarriage": bool(row["label"] == 1),
            "week": float(patient_scans["gestational_age_weeks"].max()) if int(row["label"]) == 1 else None,
        },
    }
    (patient_json_dir / f"{pid}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

print(json.dumps(validation_summary, indent=2))
display(patients_df.head())
display(scans_df.head())


## Stage 6 - Exploratory Data Analysis Plots

These plots check the generated cohort before modeling: outcome balance, scan counts, maternal risk distributions, longitudinal ultrasound trajectories, correlations, and missingness after cleaning.


In [ ]:
merged_df = scans_df.merge(patients_df[["patient_id", "label"]], on="patient_id", how="left")
merged_df["outcome"] = merged_df["label"].map({0: "Live birth", 1: "Pregnancy loss"})
patients_df["outcome"] = patients_df["label"].map({0: "Live birth", 1: "Pregnancy loss"})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.countplot(data=patients_df, x="outcome", ax=axes[0, 0], palette="Set2", hue="outcome", legend=False)
axes[0, 0].set_title("Outcome distribution")
axes[0, 0].set_xlabel("")
axes[0, 0].tick_params(axis="x", rotation=10)

scan_counts = scans_df.groupby("patient_id").size().rename("scan_count").reset_index()
sns.countplot(data=scan_counts, x="scan_count", ax=axes[0, 1], color="#4c78a8")
axes[0, 1].set_title("Scans per patient")

sns.violinplot(data=patients_df, x="outcome", y="age", ax=axes[0, 2], palette="Set2", hue="outcome", legend=False)
axes[0, 2].set_title("Maternal age by outcome")
axes[0, 2].set_xlabel("")
axes[0, 2].tick_params(axis="x", rotation=10)

sns.violinplot(data=patients_df, x="outcome", y="bmi", ax=axes[1, 0], palette="Set2", hue="outcome", legend=False)
axes[1, 0].set_title("BMI by outcome")
axes[1, 0].set_xlabel("")
axes[1, 0].tick_params(axis="x", rotation=10)

corr = merged_df[["FHR", "CRL", "GS", "YSD", "gestational_age_weeks", "label"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Feature correlation")

missing_rates = scans_df[["FHR", "CRL", "GS", "YSD"]].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "feature"})
sns.barplot(data=missing_rates, x="feature", y="missing_rate", ax=axes[1, 2], color="#59a14f")
axes[1, 2].set_ylim(0, 1)
axes[1, 2].set_title("Missingness after cleaning")

fig.tight_layout()
fig.savefig(NOTEBOOK_FIGURES_DIR / "eda_summary.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, feature in zip(axes.ravel(), ["CRL", "GS", "FHR", "YSD"]):
    sns.scatterplot(
        data=merged_df,
        x="gestational_age_weeks",
        y=feature,
        hue="outcome",
        alpha=0.35,
        s=24,
        ax=ax,
    )
    sns.lineplot(
        data=merged_df,
        x="gestational_age_weeks",
        y=feature,
        hue="outcome",
        estimator="mean",
        errorbar="sd",
        linewidth=2.4,
        legend=False,
        ax=ax,
    )
    ax.set_title(f"{feature} trajectory by outcome")
    ax.set_xlabel("Gestational age (weeks)")
fig.tight_layout()
fig.savefig(NOTEBOOK_FIGURES_DIR / "ultrasound_trajectories.png", dpi=180, bbox_inches="tight")
plt.show()


## Stage 7 - Dataset Tensors, Splits, And Smoke Test

This stage verifies the PyTorch dataset, train/validation/test split, normalization stats, temporal tensor shapes, and graph/model wiring before any longer training begins.


In [ ]:
from data.dataset import get_dataloaders

loaders, norm_stats = get_dataloaders(
    str(DATA_DIR / "patients.csv"),
    str(DATA_DIR / "scans.csv"),
    batch_size=CONFIG["batch_size"],
    max_scans=5,
    seed=SEED,
)

batch = next(iter(loaders["train"]))
print("Batch keys:", list(batch.keys()))
print("temporal_features:", tuple(batch["temporal_features"].shape))
print("gestational_ages :", tuple(batch["gestational_ages"].shape))
print("temporal_mask    :", tuple(batch["temporal_mask"].shape))
print("maternal_features:", tuple(batch["maternal_features"].shape))
print("labels           :", tuple(batch["label"].shape), "positive rate in batch", float(batch["label"].mean()))
print("Normalization stats:")
print(json.dumps({k: [round(float(v[0]), 4), round(float(v[1]), 4)] for k, v in norm_stats.items()}, indent=2))

run_command([sys.executable, "scripts/smoke_test.py"])


## Stage 8 - Train PREG-Net, FETA-Transformer, And Ensembles

PREG-Net is the primary graph-attention model. FETA-Transformer is the temporal-attention comparison model. Two late-fusion ensembles are trained: simple averaging and learned fusion.


In [ ]:
common_train_args = [
    "--patients-csv", DATA_DIR / "patients.csv",
    "--scans-csv", DATA_DIR / "scans.csv",
    "--epochs", CONFIG["train_epochs"],
    "--patience", CONFIG["train_patience"],
    "--batch-size", CONFIG["batch_size"],
    "--seed", SEED,
    "--device", DEVICE,
]

run_command([sys.executable, "scripts/train_preg.py", *common_train_args, "--out-dir", RESULTS_DIR / "preg_net"])
run_command([sys.executable, "scripts/train_feta.py", *common_train_args, "--out-dir", RESULTS_DIR / "feta_transformer"])
run_command([sys.executable, "scripts/train_ensemble.py", *common_train_args, "--out-dir", RESULTS_DIR / "ensemble"])
run_command([sys.executable, "scripts/train_ensemble.py", *common_train_args, "--out-dir", RESULTS_DIR / "ensemble_learned", "--learned-fusion"])

for name in ["preg_net", "feta_transformer", "ensemble", "ensemble_learned"]:
    metrics_path = RESULTS_DIR / name / "metrics.json"
    print("\n", name)
    display_if_exists(metrics_path)


## Stage 9 - Train Baselines And Compare Models

The baseline script builds patient-level tabular features from the same patient/scan CSVs and trains logistic regression, MLP, Random Forest, HistGradientBoosting, and XGBoost when the corresponding packages are installed.


In [ ]:
baseline_device = DEVICE if DEVICE in {"cpu", "mps", "cuda"} else "cpu"
run_command([
    sys.executable,
    "scripts/train_baselines.py",
    "--patients-csv", DATA_DIR / "patients.csv",
    "--scans-csv", DATA_DIR / "scans.csv",
    "--out-dir", RESULTS_DIR / "baselines",
    "--epochs", CONFIG["baseline_epochs"],
    "--patience", CONFIG["baseline_patience"],
    "--seed", SEED,
    "--device", baseline_device,
])

run_command([sys.executable, "scripts/compare_models.py", "--split", "test", "--out-prefix", RESULTS_DIR / "model_comparison"])
comparison_df = pd.read_csv(RESULTS_DIR / "model_comparison.csv")
display(comparison_df)

fig, ax = plt.subplots(figsize=(11, max(4, 0.42 * len(comparison_df))))
plot_df = comparison_df.sort_values("auroc", ascending=True)
ax.barh(plot_df["model"], plot_df["auroc"], color="#4c78a8", label="AUROC")
ax.barh(plot_df["model"], plot_df["f1"], color="#f58518", alpha=0.75, label="F1")
ax.set_xlim(0, 1.02)
ax.set_title("Fixed test split model comparison")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(NOTEBOOK_FIGURES_DIR / "model_comparison_bar.png", dpi=180, bbox_inches="tight")
plt.show()


## Stage 10 - Cross-Validation, Statistical Tests, And Calibration

Cross-validation reports fold-level stability. Statistical analysis adds bootstrap confidence intervals, exact McNemar tests, paired bootstrap AUROC comparisons, Brier score, ECE/MCE, and calibration curves.


In [ ]:
cv_command = [
    sys.executable,
    "scripts/cross_validate.py",
    "--patients-csv", DATA_DIR / "patients.csv",
    "--scans-csv", DATA_DIR / "scans.csv",
    "--models", CONFIG["cv_models"],
    "--n-folds", CONFIG["cv_folds"],
    "--epochs", CONFIG["cv_epochs"],
    "--patience", CONFIG["cv_patience"],
    "--batch-size", CONFIG["batch_size"],
    "--out-dir", RESULTS_DIR / "cross_validation",
    "--seed", SEED,
    "--device", DEVICE,
]
if CONFIG["cv_max_folds"] is not None:
    cv_command.extend(["--max-folds", CONFIG["cv_max_folds"]])
run_command(cv_command)

display_if_exists(RESULTS_DIR / "cross_validation" / "summary.md")

run_command([
    sys.executable,
    "scripts/statistical_analysis.py",
    "--iterations", CONFIG["bootstrap_iterations"],
    "--out-dir", RESULTS_DIR / "statistical_analysis",
    "--seed", SEED,
])

display_if_exists(RESULTS_DIR / "statistical_analysis" / "summary.md")
display_if_exists(RESULTS_DIR / "statistical_analysis" / "calibration_curves.svg")


## Stage 11 - Explainability And Paper-Ready Artifacts

This stage exports temporal attention, maternal-to-time maps, PREG-Net node/edge rankings, case-study patient trajectories/graphs, and publication-ready figures/tables.


In [ ]:
export_device = DEVICE if DEVICE in {"cpu", "mps", "cuda"} else "cpu"
run_command([
    sys.executable,
    "scripts/export_explainability.py",
    "--patients-csv", DATA_DIR / "patients.csv",
    "--scans-csv", DATA_DIR / "scans.csv",
    "--feta-dir", RESULTS_DIR / "feta_transformer",
    "--preg-dir", RESULTS_DIR / "preg_net",
    "--out-dir", RESULTS_DIR / "explainability",
    "--device", export_device,
    "--seed", SEED,
])

display_if_exists(RESULTS_DIR / "explainability" / "summary.md")

run_command([
    sys.executable,
    "scripts/generate_paper_artifacts.py",
    "--patients-csv", DATA_DIR / "patients.csv",
    "--scans-csv", DATA_DIR / "scans.csv",
    "--out-dir", RESULTS_DIR / "paper_artifacts",
    "--comparison-csv", RESULTS_DIR / "model_comparison.csv",
    "--cv-summary-csv", RESULTS_DIR / "cross_validation" / "summary.csv",
    "--ci-csv", RESULTS_DIR / "statistical_analysis" / "metric_confidence_intervals.csv",
    "--calibration-svg", RESULTS_DIR / "statistical_analysis" / "calibration_curves.svg",
    "--explainability-dir", RESULTS_DIR / "explainability",
])

manifest_path = RESULTS_DIR / "paper_artifacts" / "manifest.json"
display_if_exists(manifest_path)


## Stage 12 - Final Dashboard And Output Zip

The notebook ends by displaying the most important generated outputs and creating a zip archive of `results/` for easy download from Colab.


In [ ]:
print("Final project artifacts")
print("- Project root:", ROOT)
print("- Data:", DATA_DIR.relative_to(ROOT))
print("- Results:", RESULTS_DIR.relative_to(ROOT))
print("- Notebook figures:", NOTEBOOK_FIGURES_DIR.relative_to(ROOT))

if (RESULTS_DIR / "model_comparison.csv").exists():
    comparison_df = pd.read_csv(RESULTS_DIR / "model_comparison.csv")
    display(comparison_df)

figures_dir = RESULTS_DIR / "paper_artifacts" / "figures"
tables_dir = RESULTS_DIR / "paper_artifacts" / "tables"
for figure_name in [
    "fig_roc_curves.svg",
    "fig_precision_recall_curves.svg",
    "fig_confusion_matrices.svg",
    "fig_calibration_curves.svg",
    "fig_feta_architecture.svg",
    "fig_preg_net_architecture.svg",
    "fig_feta_temporal_attention_heatmap.svg",
    "fig_preg_node_importance.svg",
    "fig_preg_edge_attention.svg",
]:
    path = figures_dir / figure_name
    if path.exists():
        print(f"\n{figure_name}")
        display_if_exists(path)

for table_name in [
    "table_dataset_demographics_by_outcome.md",
    "table_model_comparison_metrics.md",
    "table_cross_validation_summary.md",
    "table_confidence_intervals.md",
    "table_hyperparameters.md",
]:
    path = tables_dir / table_name
    if path.exists():
        print(f"\n{table_name}")
        display_if_exists(path)

checklist = {
    "embedded_source_files": (ROOT / "scripts" / "train_preg.py").exists(),
    "download_manifest": (DATASETS_DIR / "download_manifest.json").exists(),
    "synthetic_patients_csv": (DATA_DIR / "patients.csv").exists(),
    "synthetic_scans_csv": (DATA_DIR / "scans.csv").exists(),
    "validation_summary": (DATA_DIR / "notebook_validation_summary.json").exists(),
    "preg_net_metrics": (RESULTS_DIR / "preg_net" / "metrics.json").exists(),
    "feta_metrics": (RESULTS_DIR / "feta_transformer" / "metrics.json").exists(),
    "ensemble_metrics": (RESULTS_DIR / "ensemble" / "metrics.json").exists(),
    "learned_ensemble_metrics": (RESULTS_DIR / "ensemble_learned" / "metrics.json").exists(),
    "baseline_metrics": (RESULTS_DIR / "baselines" / "metrics.json").exists(),
    "model_comparison": (RESULTS_DIR / "model_comparison.csv").exists(),
    "cross_validation": (RESULTS_DIR / "cross_validation" / "summary.csv").exists(),
    "statistical_analysis": (RESULTS_DIR / "statistical_analysis" / "summary.md").exists(),
    "explainability": (RESULTS_DIR / "explainability" / "summary.md").exists(),
    "paper_artifacts": (RESULTS_DIR / "paper_artifacts" / "manifest.json").exists(),
}
checklist_df = pd.DataFrame([{"artifact": key, "exists": value} for key, value in checklist.items()])
display(checklist_df)
if not checklist_df["exists"].all():
    missing = checklist_df.loc[~checklist_df["exists"], "artifact"].tolist()
    raise RuntimeError(f"Notebook completed but some expected artifacts are missing: {missing}")

archive_base = ROOT / f"feta_preg_results_{RUN_MODE}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULTS_DIR))
print(f"Results zip created: {archive_path}")
if IN_COLAB:
    print("Download from the Colab file browser, or run: from google.colab import files; files.download(str(archive_path))")
print("Notebook pipeline complete.")
